# 긴 조문 대응 확인: 예전 / A 전부 항 / B 조문 검색 + 항 리랭크 (Colab GPU)

**보려는 것**: 조문이 길어서 정답 항이 LLM까지 못 가던 문제(예: 취업규칙 제56조 2,117자 → 1,200자에서 잘려 ⑦⑧이 빠짐)가 항 단위 조각(DB PR #8)으로 나아지는지.

검색은 **11/18 임직원 평가 노트북(Qwen3.5-27B)과 같은 코드·같은 튜닝값**이에요.
Dense(BGE-M3) 30 + BM25(Kiwi, backend `rewrite`) 30 → 가중 RRF(법령 ×0.6) → 후보 40 → bge-reranker-v2-m3 → 5개.

> **이번 실행: 내부규정 가점 없음** (11/18은 +0.2). 가점이 붙으면 상관없는 내부규정 조문도 최소 0.2점이 돼서 거절 기준(1등 점수 < 0.204)이 사실상 작동하지 않았어요 (2026-11-19 실행 결과). 가점이 하던 "내부규정을 법령보다 위에"는 아래 '사내 우선 정확도'로 따로 확인해요. RRF 단계의 법령 ×0.6은 그대로예요.

| 방식 | ② 검색 | ③ 리랭커 | LLM에 넘기는 5개 |
| --- | --- | --- | --- |
| 예전 | 조문 조각 (2,697개) | 조문 조각 | 조문 전체(나뉜 조각 합침) + 별표, **1,200자에서 자름** |
| A 전부 항 | 항 조각 (5,720개) | 항 | 항 5개 (+별표) |
| **B 조문검색+항리랭크** | 조문 조각 (예전 그대로) | 후보 조문을 **항으로 펼쳐** 점수, 조문 점수 = 최고 항 | 조문마다 **점수 높은 항부터** 이어 붙여 1,200자 (+별표) |

지표
- **정답 항 전달률**: 골든셋 `gold_hang`(정답 항)이 LLM에 넘기는 5개 안에 **앞 30자·끝 30자 모두** 들어 있는 비율. 중간에 잘려도 실패
- **조문 recall@5·30 / MRR**: 조문 순서(중복 없음)에서 정답 조문(`gold_articles`) 위치
- **(참고) 예전 방식 그대로**: 예전 조각 + `gold_chunks` → 11/18 노션 dev 결과(rerank recall@5 0.651)가 재현되는지
- **긴 조문 문항**: 정답 조문 하나라도 원문이 1,200자를 넘는 문항만 따로
- dev(75)·test(166) 둘 다 돌려요. ⑥⑦⑧ 같은 문항(P2-7-01, P3-8-09)은 test에 있어요

- **속도**: 질문 하나당 검색 시간(p50·p95)과 리랭커가 본 쌍 수 (LLM 답변 생성은 빼고)
- **거절·되묻기 기준**: B 점수로 거절 기준값(`verify_min_score`)과 되묻기 설정을 dev로 다시 맞추고 test로 확인

런타임 → 런타임 유형 변경 → **GPU**(T4 이상) → 모두 실행. T4 기준 20~30분.

In [ ]:
!pip install -q -U "sentence-transformers>=3.4" rank_bm25 kiwipiepy python-dotenv

In [ ]:
import os, sys, json, re, time, math, subprocess, importlib.util
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, torch

ROOT = Path("/content/hr_chatbot")
ROOT.mkdir(parents=True, exist_ok=True)

# 11/18 임직원 평가(Qwen3.5-27B-FP8) summary.json 의 cfg 값 그대로
CFG = {"embed_model": "BAAI/bge-m3", "rerank_model": "BAAI/bge-reranker-v2-m3",
       "k_dense": 30, "k_bm25": 30, "k_candidates": 40, "rrf_c": 60,
       "bm25_weight": 1.0, "dense_weight": 1.0, "law_weight": 0.6, "rewrite_bm25": True,
       "internal_bonus": 0.0,   # 11/18은 0.2. 가점이 거절 기준을 망가뜨려서 뺌 (2026-11-19)
       "k_context": 5, "attach_appendix": True, "max_chars_per_article": 1200}

for repo, branch in (("backend", "main"), ("DB", "main"), ("RAG", "main")):
    d = ROOT / repo
    if d.exists():
        subprocess.run(f"rm -rf {d}", shell=True)
    subprocess.run(f"git clone -q --depth 1 -b {branch} https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git {d}", shell=True, check=True)
    print(f"{repo:11s} {branch:30s}", subprocess.run(f"git -C {d} log -1 --format='%h %ad %s' --date=short", shell=True,
                                                     capture_output=True, text=True).stdout.strip())
BACKEND, DB, HYERIM = ROOT / "backend", ROOT / "DB", ROOT / "RAG"

# 팀 채점 스크립트 (eg.retrieval_metrics, eg.get_splits …)
os.environ["BACKEND_DIR"] = str(BACKEND)
spec = importlib.util.spec_from_file_location("eval_golden", HYERIM / "scripts/eval_golden.py")
eg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eg)
ITEMS = eg.load_items()
SPLIT = eg.get_splits(ITEMS)

# backend 검색어 보강(glossary) = 설계도 ①의 search_query. DB 없이 불러옴
os.environ.update({"HRDB_URL": "", "SEARCH_MODE": "keyword", "LLM_BASE_URL": "", "PSEUDO_KEY": "colab-notebook-only"})
sys.path.insert(0, str(BACKEND))
from app.graph import retriever as be_retriever
print(f"골든셋 {len(ITEMS)}문항 · 재작성 예: {be_retriever.rewrite('결혼하면 경조휴가 며칠이야?')}")

In [ ]:
def jl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

def _cite(r):
    art, t = (r.get("article") or "").strip(), (r.get("article_title") or "").strip()
    return f"{r['doc_name']} {art}" + (f"({t[:20]})" if t and art.startswith("제") else "")

def load_corpus(paths):   # 11/18 노트북과 같은 형식
    out = []
    for p in paths:
        for r in jl(p):
            if not r.get("searchable"):
                continue
            out.append({"id": r["chunk_id"], "doc_id": r["document_id"], "doc_name": r["doc_name"],
                        "doc_kind": r.get("doc_kind") or ("법령" if r["chunk_id"].startswith("L") else "내부규정"),
                        "article": (r.get("article") or "").strip(), "article_title": (r.get("article_title") or "").strip(),
                        "part": r.get("part") or 1, "citation": _cite(r), "text": r["text"],
                        "article_id": r.get("article_id"), "hang": r.get("hang", ""),
                        "embed_text": f"{r['doc_name']} {r.get('article') or ''} {r.get('article_title') or ''}".strip() + "\n" + r["text"]})
    return out

CORPORA = {"old": load_corpus([DB / "data/regs/chunks.jsonl", DB / "data/laws/chunks.jsonl"]),
           "new": load_corpus([DB / "data/hang/chunks.jsonl"])}
O2N = json.load(open(DB / "data/hang/old_to_new.json", encoding="utf-8"))
ART = {k: v["article_id"] for k, v in O2N.items()} | {c["id"]: c["article_id"] for c in CORPORA["new"]}   # 조각 → 조문
ART_LEN = {a["article_id"]: len(a["text"]) for a in jl(DB / "data/hang/articles.jsonl")}                   # 조문 원문 길이
NEW_TEXT = {c["id"]: c for c in CORPORA["new"]}
for k, v in CORPORA.items():
    print(k, len(v), "조각, 평균", int(np.mean([len(c["text"]) for c in v])), "자")

In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음 (느려요)")

def _load(cls, name, **kw):
    try:
        return cls(name, device=DEVICE, model_kwargs={"torch_dtype": torch.float16 if HAS_GPU else torch.float32}, **kw)
    except TypeError:
        m = cls(name, device=DEVICE, **kw)
        if HAS_GPU:
            m.model.half() if hasattr(m, "model") else m.half()
        return m

kiwi = Kiwi()
_KEEP = {"NNG", "NNP", "NR", "SL", "SN", "SH", "XR", "VV", "VA"}
def ko_tokens(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)
    return [t.form + ("다" if t.tag in ("VV", "VA") else "") for t in kiwi.tokenize(text) if t.tag in _KEEP]

EMBEDDER = _load(SentenceTransformer, CFG["embed_model"])
EMBEDDER.max_seq_length = 1024
RERANKER = _load(CrossEncoder, CFG["rerank_model"], max_length=1024)

IDX = {}
for name, chunks in CORPORA.items():
    t0 = time.time()
    bm = BM25Okapi([ko_tokens(c["embed_text"]) for c in chunks])
    emb = EMBEDDER.encode([c["embed_text"] for c in chunks], batch_size=32, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    art = defaultdict(list)
    for i, c in enumerate(chunks):
        if c["article"]:
            art[(c["doc_id"], c["article"])].append(i)
    for k in art:
        art[k].sort(key=lambda j: chunks[j]["part"])
    IDX[name] = {"CHUNKS": chunks, "BM25": bm, "DOC_EMB": emb, "ARTICLE": art}
    print(f"{name}: BM25 + 임베딩 {time.time() - t0:.0f}초")

def use(name):
    # 아래 검색 함수는 11/18 노트북 코드 그대로라 전역 변수를 씀 → 코퍼스를 바꿔 끼움
    global CHUNKS, BM25, DOC_EMB, ARTICLE, CORPUS
    CORPUS = name
    CHUNKS, BM25, DOC_EMB, ARTICLE = (IDX[name][k] for k in ("CHUNKS", "BM25", "DOC_EMB", "ARTICLE"))

In [ ]:
# ── 11/18 임직원 평가 노트북의 ②③ 코드 그대로 (바뀐 곳은 ★ 표시) ──
_QCACHE, _RCACHE = {}, {}
RR = {"pairs": 0, "sec": 0.0}     # 속도 재기용: 리랭커가 본 (질문, 조각) 쌍 수와 걸린 시간

def embed_queries(texts):
    todo = [t for t in dict.fromkeys(texts) if t not in _QCACHE]
    if todo:
        clean = [re.sub(r"\[[^\]]*\]", " ", t) for t in todo]
        for t, v in zip(todo, EMBEDDER.encode(clean, batch_size=64, normalize_embeddings=True, convert_to_numpy=True)):
            _QCACHE[t] = v.astype(np.float32)
    return np.stack([_QCACHE[t] for t in texts])

def bm25_rank(query, n):
    scores = BM25.get_scores(ko_tokens(query))
    top = np.argsort(-scores)[:n]
    return [int(i) for i in top if scores[i] > 0]

def dense_rank(query, n):
    sims = DOC_EMB @ embed_queries([query])[0]
    return [int(i) for i in np.argsort(-sims)[:n]]

def fuse(ranked_lists, weights, n, law_weight=1.0, c=None):
    c = c or CFG["rrf_c"]
    score = defaultdict(float)
    for lst, w in zip(ranked_lists, weights):
        for r, i in enumerate(lst):
            score[i] += w / (c + r + 1)
    for i in score:
        if CHUNKS[i]["doc_kind"] == "법령":
            score[i] *= law_weight
    return sorted(score, key=score.get, reverse=True)[:n]

def candidates(question, n=None):
    n = n or CFG["k_candidates"]
    bq = be_retriever.rewrite(question) if CFG["rewrite_bm25"] else question
    lists = [bm25_rank(bq, max(CFG["k_bm25"], n)), dense_rank(question, max(CFG["k_dense"], n))]
    return fuse(lists, [CFG["bm25_weight"], CFG["dense_weight"]], n, CFG["law_weight"])

def rerank_raw(query, idxs, batch_size=16):
    q = re.sub(r"\[[^\]]*\]", " ", query)
    todo = [i for i in idxs if (CORPUS, q, i) not in _RCACHE]          # ★ 캐시 키에 코퍼스 이름
    if todo:
        t_rr = time.perf_counter()
        s = np.asarray(RERANKER.predict([(q, CHUNKS[i]["embed_text"]) for i in todo], batch_size=batch_size,
                                        convert_to_numpy=True, show_progress_bar=False), dtype=float).reshape(-1)
        if s.min() < 0 or s.max() > 1:
            s = 1 / (1 + np.exp(-s))
        _RCACHE.update({(CORPUS, q, i): float(v) for i, v in zip(todo, s)})
        RR["pairs"] += len(todo)
        RR["sec"] += time.perf_counter() - t_rr
    return {i: _RCACHE[(CORPUS, q, i)] for i in idxs}

def rerank(query, idxs, bonus=None):
    bonus = CFG["internal_bonus"] if bonus is None else bonus
    raw = rerank_raw(query, idxs)
    scored = [(i, min(1.0, s + (bonus if CHUNKS[i]["doc_kind"] == "내부규정" else 0.0))) for i, s in raw.items()]
    return sorted(scored, key=lambda x: -x[1])

def search(question, mode="rerank", n=None):
    n = n or CFG["k_candidates"]
    if mode == "bm25":
        q = be_retriever.rewrite(question) if CFG["rewrite_bm25"] else question
        return [(i, 1 / (r + 1)) for r, i in enumerate(bm25_rank(q, n))]
    if mode == "dense":
        return [(i, 1 / (r + 1)) for r, i in enumerate(dense_rank(question, n))]
    cand = candidates(question, n)
    if mode == "hybrid":
        return [(i, 1 / (r + 1)) for r, i in enumerate(cand)]
    return rerank(question, cand)

def expand(i, score):
    c = CHUNKS[i]
    # ★ 예전 조각: 같은 조문의 나뉜 조각을 합침(11/18과 같음) / 새 조각: 항 그대로 (합치면 다시 길어짐)
    parts = ARTICLE.get((c["doc_id"], c["article"]), [i]) if CORPUS == "old" and c["article"] else [i]
    ids, text = [CHUNKS[j]["id"] for j in parts], "\n".join(CHUNKS[j]["text"] for j in parts)
    if CFG["attach_appendix"] and not c["article"].startswith("별표"):
        for n_ in dict.fromkeys(re.findall(r"별표\s*(\d+)", text)):
            appx = ARTICLE.get((c["doc_id"], f"별표 {n_}"))
            if appx and CHUNKS[appx[0]]["id"] not in ids:
                ids += [CHUNKS[j]["id"] for j in appx]
                text += f"\n[참조 별표 {n_}]\n" + "\n".join(CHUNKS[j]["text"] for j in appx)
                break
    cap = CFG["max_chars_per_article"]
    return {"id": c["id"], "ids": ids, "citation": c["citation"], "doc_kind": c["doc_kind"],
            "text": text if len(text) <= cap else text[:cap] + " …(생략)", "score": round(score, 4)}

def context(ranked, k=None):
    out, seen = [], set()
    for i, s in ranked:
        # ★ 예전: 조문마다 하나 / 새: 항마다 하나
        key = (CHUNKS[i]["doc_id"], CHUNKS[i]["article"] or CHUNKS[i]["id"]) if CORPUS == "old" else CHUNKS[i]["id"]
        if key in seen:
            continue
        seen.add(key)
        out.append(expand(i, s))
        if len(out) >= (k or CFG["k_context"]):
            break
    return out

for name in CORPORA:
    use(name)
    print(f"--- {name}")
    for h in context(search("배우자 출산휴가 며칠이야?"), 3):
        print(f"{h['score']:.3f} {h['id']} {h['citation']} · {len(h['text'])}자")

## B. 조문 검색 + 항 리랭크

```
② 하이브리드 검색: 예전 조문 단위 조각 그대로 (위와 같음) → 후보 40조각 → 조문으로 묶음
③ 리랭커: 후보 조문을 항으로 펼쳐 항마다 점수 (+내부규정 가점)
   → 조문 점수 = 그 조문에서 가장 높은 항 점수 → 조문 순서
   → LLM에 넘길 때: 조문마다 점수 높은 항부터 이어 붙이고(+별표) 1,200자에서 자름 → 5개
```

In [ ]:
OLDC, NEWC = IDX["old"]["CHUNKS"], IDX["new"]["CHUNKS"]
NEW_BY_ART = defaultdict(list)                       # 조문 → 새 항 조각 번호 (조문 안 순서)
for j, c in enumerate(NEWC):
    NEW_BY_ART[c["article_id"]].append(j)
ART_TEXT = {a["article_id"]: a["text"] for a in jl(DB / "data/hang/articles.jsonl")}
OLD_ARTICLE = IDX["old"]["ARTICLE"]

def dedupe(xs): return list(dict.fromkeys(xs))

def squash(s): return re.sub(r"\s+", "", s)

def head_of(c): return f"{c['article']}({c['article_title']})" if c["article_title"] else c["article"]

def search_B(question):
    use("old")
    arts = dedupe(ART[OLDC[i]["id"]] for i in candidates(question))
    use("new")
    idxs = [j for a in arts for j in NEW_BY_ART.get(a, [])]
    sc = dict(rerank(question, idxs))
    arts = [a for a in arts if NEW_BY_ART.get(a)]
    art_rank = sorted(arts, key=lambda a: -max(sc[j] for j in NEW_BY_ART[a]))
    return art_rank, sc

def context_B(art_rank, sc, k=None):
    out = []
    for a in art_rank[:k or CFG["k_context"]]:
        js = sorted(NEW_BY_ART[a], key=lambda j: -sc[j])
        parts = []
        for n_, j in enumerate(js):
            t, h = NEWC[j]["text"], head_of(NEWC[j])
            parts.append(t[len(h):].lstrip() if n_ and t.startswith(h) else t)
        text, c0 = "\n".join(parts), NEWC[js[0]]
        if CFG["attach_appendix"] and not c0["article"].startswith("별표"):
            for n_ in dict.fromkeys(re.findall(r"별표\s*(\d+)", ART_TEXT.get(a, text))):
                appx = OLD_ARTICLE.get((c0["doc_id"], f"별표 {n_}"))
                if appx and ART.get(OLDC[appx[0]]["id"]) != a:
                    text += f"\n[참조 별표 {n_}]\n" + "\n".join(OLDC[x]["text"] for x in appx)
                    break
        cap = CFG["max_chars_per_article"]
        out.append({"id": a, "text": text if len(text) <= cap else text[:cap] + " …(생략)", "score": round(max(sc[j] for j in js), 4)})
    return out

art_rank, sc = search_B("며느리가 출산하면 할아버지인 저도 휴가 있어요?")
for h in context_B(art_rank, sc, 3):
    print(f"{h['score']:.3f} {h['id']} · {len(h['text'])}자 · {squash(h['text'])[:80]}")

In [ ]:
def hang_sig(cid):
    # 정답 항 글자(머리말 제N조(제목) 뺌)의 앞 30자와 끝 30자 → 둘 다 LLM에 넘긴 글에 있어야 전달됨 (중간에 잘리면 실패)
    c = NEW_TEXT[cid]
    body, h = c["text"], head_of(c)
    s = squash(body[len(h):] if body.startswith(h) else body)
    return (s[:30], s[-30:]) if len(s) > 30 else (s, s)

def delivered(cid, blob):
    a, b = hang_sig(cid)
    return a in blob and b in blob

def art_metrics(gold, arts):
    # arts: 조문 순서 (중복 없음)
    g = set(gold)
    hits = [i for i, x in enumerate(arts) if x in g]
    return {"조문 recall@5": sum(i < 5 for i in hits) / len(g), "조문 recall@30": sum(i < 30 for i in hits) / len(g),
            "조문 MRR": 1 / (hits[0] + 1) if hits else 0.0}

def is_long(it):
    return any(ART_LEN.get(a, 0) > CFG["max_chars_per_article"] for a in it["gold_articles"]["primary"])

METHODS = {"예전": "old", "A 전부 항": "new", "B 조문검색+항리랭크": "B"}
SPLITS = ["dev", "test"]
ROWS = []
for split in SPLITS:
    QS = [it for it in ITEMS if SPLIT[it["id"]] == split and it["gold_chunks"]["primary"]]
    for label, m in METHODS.items():
        def run_one(question):
            if m == "B":
                arts, sc = search_B(question)
                return arts, context_B(arts, sc), None
            use(m)
            ranked = search(question, "rerank")
            got = [CHUNKS[i]["id"] for i, _ in ranked][:30]
            return dedupe(ART[x] for x in got), context(ranked), got
        run_one("연차는 며칠이야?")                     # 예열 (GPU 첫 호출은 느려서 빼고 잼)
        t0 = time.time()
        for it in QS:
            # 속도: 캐시를 비우고 질문 하나씩 처음부터 (질문 임베딩 → ② 검색 → ③ 리랭커 → 넘길 글 만들기)
            _QCACHE.clear(); _RCACHE.clear(); RR.update(pairs=0, sec=0.0)
            if HAS_GPU:
                torch.cuda.synchronize()
            tq = time.perf_counter()
            arts, ctx, got = run_one(it["question"])
            if HAS_GPU:
                torch.cuda.synchronize()
            ms = (time.perf_counter() - tq) * 1000
            blob = squash("\n".join(h["text"] for h in ctx))
            gh = it["gold_hang"]
            hang_ok = gh["auto"] is True and bool(gh["primary"])
            row = {"split": split, "방식": label, "id": it["id"], "question": it["question"], "long": is_long(it),
                   "ctx_chars": sum(len(h["text"]) for h in ctx), "ctx": [h["id"] for h in ctx],
                   "cut": sum(h["text"].endswith("…(생략)") for h in ctx),
                   "ms": ms, "rerank_ms": RR["sec"] * 1000, "rerank_pairs": RR["pairs"],
                   "정답 항 전달률": np.mean([delivered(c, blob) for c in gh["primary"]]) if hang_ok else np.nan,
                   **art_metrics(it["gold_articles"]["primary"], arts)}
            if m == "old":
                row |= {f"(예전) {k}": v for k, v in eg.retrieval_metrics(it["gold_chunks"]["primary"], got, it["document_ids"]).items()
                        if k in ("recall@5", "recall@30", "mrr")}
            ROWS.append(row)
        print(f"{split} {label}: {len(QS)}문항 {time.time() - t0:.0f}초")
R = pd.DataFrame(ROWS)

In [ ]:
KEYS = ["정답 항 전달률", "조문 recall@5", "조문 recall@30", "조문 MRR", "ctx_chars", "cut"]
def summary(df):
    g = df.groupby("방식", sort=False)
    s = g[KEYS].mean().round(3).reindex(list(METHODS))
    s["문항"] = g.size()
    s["항 전달률 문항"] = g["정답 항 전달률"].count()
    return s.rename(columns={"ctx_chars": "LLM 입력 글자수", "cut": "잘린 조문 수(5개 중)"})

for split in SPLITS:
    D = R[R["split"] == split]
    print(f"\n========== {split} ==========")
    print(f"■ 긴 조문 문항 (정답 조문 원문 > {CFG['max_chars_per_article']}자)")
    display(summary(D[D["long"]]))
    print("■ 나머지 문항")
    display(summary(D[~D["long"]]))
    print(f"■ 전체 {split}")
    display(summary(D))
print("\n■ (참고) 예전 조각 + 예전 정답(gold_chunks) — 11/18 노션 dev rerank: recall@5 0.651 · recall@30 0.782 · MRR 0.612")
display(R[R["방식"] == "예전"].groupby("split")[[c for c in R.columns if c.startswith("(예전)")]].mean().round(3))

## 속도 (질문 하나당, LLM 답변 생성 제외)
캐시를 비우고 질문마다 처음부터 쟀어요: 질문 임베딩 → ② 검색(BM25·Dense·RRF) → ③ 리랭커 → LLM에 넘길 글 만들기. GPU 첫 호출(예열)은 뺐어요.
- B는 후보 조문을 항으로 펼쳐 리랭커가 보는 쌍이 늘어요 (예전·A는 40쌍)
- 백엔드 실제 응답시간 목표: p50 ≤ 10초 (노션 11/18: 임직원 p50 36.1초, 대부분 LLM 답변 생성)

In [ ]:
def speed(df):
    g = df.groupby("방식", sort=False)
    out = pd.DataFrame({"p50 ms": g["ms"].median(), "p95 ms": g["ms"].quantile(0.95), "평균 ms": g["ms"].mean(),
                        "리랭커 평균 ms": g["rerank_ms"].mean(), "리랭커 쌍 수 평균": g["rerank_pairs"].mean(),
                        "리랭커 쌍 수 최대": g["rerank_pairs"].max()}).reindex(list(METHODS))
    out["예전 대비 p50"] = (out["p50 ms"] / out.loc["예전", "p50 ms"]).map(lambda x: f"×{x:.2f}")
    return out.round(1)

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음")
for split in SPLITS:
    print(f"■ {split}")
    display(speed(R[R["split"] == split]))
print("■ dev + test")
display(speed(R))

## 거절·되묻기 기준 다시 맞추기
11/18 파이프라인의 두 규칙은 **예전 방식 점수(리랭커가 조 덩어리를 볼 때)**로 정한 값이에요. B는 리랭커가 짧은 항을 보니까 점수 범위가 달라서 다시 맞춰요.

| 규칙 | 11/18 값 | 정하는 법 (11/18과 같음) |
| --- | --- | --- |
| 거절 `verify_min_score`: 리랭커 1등 점수가 이보다 낮으면 "규정에서 확인되지 않아요" | 0.204 (가점 0.2 포함이라 원래 점수로는 0.004) | dev **답할 수 있는 문항**의 1등 점수 아래 5% 지점 (잘못된 거절 ≤ 5%) |
| 되묻기 `clarify`: 질문이 짧고, 1등과 비슷한 점수의 주제가 여러 개면 되묻기 | margin 0.2 · 주제 2개↑ · 12자↓ · 1등 < 0.8 | dev 일반 문항 불필요 되묻기 ≤ 5% 중 되묻기 재현율 최고 (`rerank_cases` clarify) |

- dev로 정하고 test로 확인해요. "B에 11/18 값을 그대로 쓰면?" 줄도 같이 봐요 (걱정했던 경우)
- ⚠️ 골든셋 '규정 없음' 문항이 dev 3 · test 7개뿐이라 거절률은 참고만. **AUC**(답할 문항과 규정 없음 문항을 1등 점수로 얼마나 잘 가르는지, dev+test 합쳐서 · 1이면 완벽 · 0.5면 못 가름)를 같이 봐요

In [ ]:
def ranked_scored(method, q):
    # [(주제, 점수)] 높은 순. 예전·A: 조각, B: 조문(점수 = 최고 항)
    if method == "B":
        arts, sc = search_B(q)
        out = []
        for a in arts:
            c = NEWC[NEW_BY_ART[a][0]]
            out.append((c["article_title"] or c["doc_name"], max(sc[j] for j in NEW_BY_ART[a])))
        return out
    use(method)
    return [(CHUNKS[i]["article_title"] or CHUNKS[i]["doc_name"], s) for i, s in search(q, "rerank")]

def topics(ranked, margin):
    if not ranked:
        return []
    top, out = ranked[0][1], []
    for t, s in ranked[:8]:
        if s < top - margin:
            break
        t = re.sub(r"\(.*", "", t).strip()
        if t and t not in out:
            out.append(t)
    return out

def should_clarify(question, ranked, cfg):
    if not cfg or not ranked:
        return False
    short = len(re.sub(r"\[[^\]]*\]|\s", "", question)) <= cfg["max_len"]
    return bool(short and ranked[0][1] < cfg["top_max"] and len(topics(ranked, cfg["margin"])) >= cfg["min_topics"])

def auc(pos, neg):   # pos 점수가 neg 보다 높을 확률
    if not pos or not neg:
        return np.nan
    return np.mean([(p > n) + 0.5 * (p == n) for p in pos for n in neg])

CFG_1118 = {"verify_min_score": 0.204, "clarify": {"margin": 0.2, "min_topics": 2, "max_len": 12, "top_max": 0.8}}
RERANK_CASES = [json.loads(l) for l in open(HYERIM / "eval/golden/rerank_cases.jsonl", encoding="utf-8") if l.strip()]
CAL_ITEMS = [it for it in ITEMS if SPLIT[it["id"]] in ("dev", "test")]           # 규정 없음·틀린 전제 등 전부
CL_CASES = [c for c in RERANK_CASES if c["kind"] == "clarify"]
CAL = {}
t0 = time.time()
for label, m in METHODS.items():
    CAL[label] = {"items": [(it, ranked_scored(m, it["question"])) for it in CAL_ITEMS],
                  "clarify": [(c, ranked_scored(m, c["question"])) for c in CL_CASES]}
    print(f"{label}: {time.time() - t0:.0f}초")

def verify_rows(label, thr, name):
    rows = []
    for split in ("dev", "test"):
        X = [(it, r) for it, r in CAL[label]["items"] if SPLIT[it["id"]] == split]
        ans = [r[0][1] if r else 0.0 for it, r in X if it["type"] != "규정 없음"]
        neg = [r[0][1] if r else 0.0 for it, r in X if it["type"] == "규정 없음"]
        rows.append({"방식": label, "기준": name, "기준값": round(thr, 3), "split": split,
                     "잘못된 거절률(목표 ≤0.05)": round(np.mean([s < thr for s in ans]), 3),
                     "규정 없음 거절률(목표 ≥0.9)": round(np.mean([s < thr for s in neg]), 3) if neg else np.nan,
                     "규정 없음 문항": len(neg)})
    return rows

VER, THR = [], {}
for label in METHODS:
    dev_ans = [r[0][1] if r else 0.0 for it, r in CAL[label]["items"] if SPLIT[it["id"]] == "dev" and it["type"] != "규정 없음"]
    THR[label] = float(np.quantile(dev_ans, 0.05))
    VER += verify_rows(label, THR[label], "dev로 다시 맞춤")
    if label != "예전":
        VER += verify_rows(label, CFG_1118["verify_min_score"], "11/18 값 그대로")
VER = pd.DataFrame(VER)
print("■ 거절 기준")
display(VER.pivot_table(index=["방식", "기준", "기준값"], columns="split",
                        values=["잘못된 거절률(목표 ≤0.05)", "규정 없음 거절률(목표 ≥0.9)"], sort=False))

SEP = []
for label in METHODS:
    ans = [r[0][1] if r else 0.0 for it, r in CAL[label]["items"] if it["type"] != "규정 없음"]
    neg = [r[0][1] if r else 0.0 for it, r in CAL[label]["items"] if it["type"] == "규정 없음"]
    q = np.quantile(ans, [0.05, 0.5])
    SEP.append({"방식": label, "AUC (dev+test)": round(auc(ans, neg), 3), "답할 문항 1등 점수 5%": round(q[0], 3),
                "답할 문항 중앙값": round(q[1], 3), "규정 없음 1등 점수 (전부)": ", ".join(f"{s:.2f}" for s in sorted(neg))})
print("■ 1등 점수로 답할 문항과 규정 없음 문항을 가르는 힘")
display(pd.DataFrame(SEP).set_index("방식"))

In [ ]:
# 가점이 하던 일 확인: 사내 규정과 법이 둘 다 맞을 때 사내 규정이 위에 오나 / 함정 조문(임원·국외출장 등)이 상위 5개에 섞이나
def art_order(method, q):
    if method == "B":
        return search_B(q)[0]
    use(method)
    return dedupe(ART[CHUNKS[i]["id"]] for i, _ in search(q, "rerank"))

RC = []
for label, m in METHODS.items():
    for c in RERANK_CASES:
        if c["kind"] not in ("internal_first", "wrong_target"):
            continue
        arts = art_order(m, c["question"])
        pos = lambda ids: min([arts.index(ART[x]) for x in ids if x in ART and ART[x] in arts] or [999])
        if c["kind"] == "internal_first":
            RC.append({"방식": label, "split": c["split"], "지표": "사내 우선 정확도", "값": pos(c["gold_chunks"]["primary"]) < pos(c["rank_below"])})
        else:
            RC.append({"방식": label, "split": c["split"], "지표": "함정 조문 혼입률", "값": pos(c["must_not_chunks"]) < 5})
RC = pd.DataFrame(RC)
print("■ 사내 우선(internal_first dev 1 · test 2) · 함정 조문(wrong_target dev 3 · test 4) — 11/18(가점 0.2) 노션: 함정 혼입 0건")
display(RC.pivot_table(index="방식", columns=["지표", "split"], values="값", aggfunc="mean", sort=False).round(3))

In [ ]:
# 되묻기: 11/18 노트북과 같은 격자에서 dev로 고르고 test로 확인
GRID = [{"margin": a, "min_topics": b, "max_len": c, "top_max": d}
        for a in (0.05, 0.1, 0.2, 0.3) for b in (2, 3) for c in (12, 18, 25) for d in (0.5, 0.8, 1.01)]

def clarify_metrics(label, cfg, split):
    cl = [(c, r) for c, r in CAL[label]["clarify"] if c["split"] == split]
    normal = [(it, r) for it, r in CAL[label]["items"] if SPLIT[it["id"]] == split and it["type"] == "일반"]
    return {"되묻기 재현율": np.mean([should_clarify(c["question"], r, cfg) for c, r in cl]),
            "불필요 되묻기(목표 ≤0.05)": np.mean([should_clarify(it["question"], r, cfg) for it, r in normal])}

CLA, CHOSEN = [], {}
for label in METHODS:
    g = pd.DataFrame([{**cfg, **clarify_metrics(label, cfg, "dev")} for cfg in GRID])
    ok = g[g["불필요 되묻기(목표 ≤0.05)"] <= 0.05].sort_values(["되묻기 재현율", "불필요 되묻기(목표 ≤0.05)"], ascending=[False, True])
    CHOSEN[label] = ({k: ok.iloc[0][k].item() for k in ("margin", "min_topics", "max_len", "top_max")}
                     if len(ok) and ok.iloc[0]["되묻기 재현율"] > 0 else False)
    tries = [("dev로 다시 맞춤", CHOSEN[label])] + ([("11/18 값 그대로", CFG_1118["clarify"])] if label != "예전" else [])
    for name, cfg in tries:
        for split in ("dev", "test"):
            CLA.append({"방식": label, "기준": name, "설정": str(cfg), "split": split,
                        **{k: round(v, 3) for k, v in clarify_metrics(label, cfg, split).items()}})
CLA = pd.DataFrame(CLA)
print("■ 되묻기 기준 (되묻기 문항 dev 7 · test 13)")
display(CLA.pivot_table(index=["방식", "기준", "설정"], columns="split",
                        values=["되묻기 재현율", "불필요 되묻기(목표 ≤0.05)"], sort=False))
print("B 최종 기준:", {"verify_min_score": round(THR["B 조문검색+항리랭크"], 3), "clarify": CHOSEN["B 조문검색+항리랭크"]})

In [ ]:
# 긴 조문 문항 한 줄씩: 정답 항이 LLM까지 갔나 (예전 / A / B)
pd.set_option("display.max_colwidth", 60)
for split in SPLITS:
    D = R[(R["split"] == split) & R["long"]]
    L = D.pivot(index="id", columns="방식", values="정답 항 전달률")[list(METHODS)]
    L = L.join(D[D["방식"] == "예전"].set_index("id")[["question"]])
    o, a, b = (L[k] for k in METHODS)
    print(f"\n{split} 긴 조문 {len(L)}문항 (항 정답 있는 {o.notna().sum()}) · 예전 대비 "
          f"A 좋아짐 {(a > o).sum()} / 나빠짐 {(a < o).sum()} · B 좋아짐 {(b > o).sum()} / 나빠짐 {(b < o).sum()}")
    display(L[o.notna()].sort_values(list(METHODS)[2]))

print("\n제56조 뒤쪽 항(⑥⑦⑧) 문항 — 예전 방식에서 1,200자 자르기로 빠지던 것")
display(R[R["id"].isin(["P2-7-01", "P3-8-09"])].pivot(index="id", columns="방식", values="정답 항 전달률")[list(METHODS)])

In [ ]:
from google.colab import files
out = {"cfg": CFG, "methods": METHODS,
       "summary": {s: {"long": summary(R[(R["split"] == s) & R["long"]]).to_dict(),
                       "rest": summary(R[(R["split"] == s) & ~R["long"]]).to_dict(),
                       "all": summary(R[R["split"] == s]).to_dict()} for s in SPLITS},
       "speed": speed(R).to_dict(), "gpu": torch.cuda.get_device_name(0) if HAS_GPU else "cpu",
       "verify": VER.to_dict(orient="records"), "separation": SEP, "clarify": CLA.to_dict(orient="records"),
       "rerank_cases": RC.to_dict(orient="records"),
       "chosen": {k: {"verify_min_score": round(THR[k], 3), "clarify": CHOSEN[k]} for k in METHODS},
       "old_exact": R[R["방식"] == "예전"].groupby("split")[[c for c in R.columns if c.startswith("(예전)")]].mean().round(3).to_dict(),
       "rows": json.loads(R.to_json(orient="records", force_ascii=False))}
fn = "hang_long_article_dev_test_nobonus.json"
json.dump(out, open(fn, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
files.download(fn)